# 23. pgvector 양자화 — halfvec · 이진 양자화 + 재정렬

pgvector 에는 PQ 가 없다. 대신 **스칼라 양자화**로 인덱스를 줄인다.

| 방식 | 차원당 | 128차원 한 개 | pgvector |
|---|---|---|---|
| float32 (vector) | 4 B | 512 B | `vector` |
| float16 (halfvec) | 2 B | 256 B | `emb::halfvec(128)` 표현식 인덱스 |
| 1비트 (bit) | 1/8 B | 16 B | `binary_quantize(emb)::bit(128)` + 해밍 거리 `<~>` |
| PQ (13번, m=16) | – | 16 B | 없음 (FAISS 등) |

인덱스는 압축본으로 만들고, **테이블에는 원본이 그대로** 있으므로 원본으로 재정렬할 수 있다.

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결 · 인덱스 정리 |
| 3 | 이진 양자화는 무엇을 남기나 |
| 4 | 세 가지 인덱스 빌드 |
| 5 | recall — 압축본만 vs 원본 재정렬 |
| 6 | 뒷정리 |

**Run → Run All Cells** 로 한 번에 실행하거나, 셀을 하나씩 **Shift + Enter**

### 셀 1 · 설정

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"

### 셀 2 · 연결 · 인덱스 정리

In [ ]:
# 셀 2 · 연결 · 인덱스 정리
from labutil import *

conn = connect(DB_URL)
for ds in ["clu", "iso"]:
    for kind in ["hnsw", "ivf", "hnsw_m", "half", "bin"]:
        conn.execute(f"DROP INDEX IF EXISTS vec_{ds}_{kind}")

## 1. 이진 양자화는 무엇을 남기나
`binary_quantize` 는 차원마다 **0보다 크면 1, 아니면 0**. 거리는 다른 비트 수(해밍 거리)로 잰다.

### 셀 3 · 이진 양자화는 무엇을 남기나

In [ ]:
# 셀 3 · 이진 양자화는 무엇을 남기나
X = np.stack(q(conn, "SELECT emb FROM vec_clu ORDER BY id LIMIT 3000")["emb"].to_numpy())
Qv = np.stack(q(conn, "SELECT emb FROM vec_query WHERE ds='clu' ORDER BY qid LIMIT 1")["emb"].to_numpy())[0]
bits = X > 0
qb = Qv > 0

fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
axes[0].bar(range(32), X[0, :32], color=np.where(bits[0, :32], "#FF7043", "#3D484A"))
for i in range(32):
    axes[0].text(i, 0.05 if X[0, i] < 0 else -0.05, "1" if bits[0, i] else "0", ha="center",
                 va="bottom" if X[0, i] < 0 else "top", fontsize=9)
axes[0].axhline(0, color="black", lw=0.8); axes[0].set_xlabel("차원 (앞 32개)")
axes[0].set_title("float 값 → 부호만 남긴다: 4바이트가 1비트로")
true_d = np.linalg.norm(X - Qv, axis=1)
ham = (bits != qb).sum(1)
axes[1].scatter(true_d, ham, s=3, alpha=0.4, color="#3D484A")
near = np.argsort(true_d)[:30]
axes[1].scatter(true_d[near], ham[near], s=12, color="#FF7043", label="진짜 가까운 30개")
axes[1].set_xlabel("실제 L2 거리"); axes[1].set_ylabel("해밍 거리 (다른 비트 수)"); axes[1].legend()
axes[1].set_title(f"상관계수 {np.corrcoef(true_d, ham)[0, 1]:.2f} — 대략은 맞지만 거칠다")
plt.tight_layout(); plt.show()

## 2. 세 가지 인덱스 빌드

### 셀 4 · 세 가지 인덱스 빌드

In [ ]:
# 셀 4 · 세 가지 인덱스 빌드
builds = {}
specs = {
    "vector (float32)": ("hnsw", "emb vector_l2_ops"),
    "halfvec (float16)": ("half", "(emb::halfvec(128)) halfvec_l2_ops"),
    "bit (1비트)": ("bin", "(binary_quantize(emb)::bit(128)) bit_hamming_ops"),
}
for name, (kind, expr) in specs.items():
    with timer() as t:
        conn.execute(f"CREATE INDEX vec_clu_{kind} ON vec_clu USING hnsw ({expr})")
    builds[name] = {"빌드(s)": t["s"], "인덱스 MB": rel_size(conn, f"vec_clu_{kind}")}
builds = pd.DataFrame(builds).T

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
colors = ["#3D484A", "#009CA6", "#FF7043"]
for ax, col in zip(axes, ["인덱스 MB", "빌드(s)"]):
    ax.barh(builds.index, builds[col], color=colors)
    for i, v in enumerate(builds[col]):
        ax.text(v, i, f" {v:.1f}", va="center")
    ax.invert_yaxis(); ax.set_title(col)
plt.tight_layout(); plt.show()

**관찰:** 이진 인덱스는 float32 인덱스보다 몇 배 작다. 32배가 아닌 이유: 이웃 목록(그래프)과 행 주소는 그대로이기 때문.

## 3. recall — 압축본만 vs 원본으로 재정렬

### 셀 5 · recall — 압축본만 vs 원본 재정렬

In [ ]:
# 셀 5 · recall — 압축본만 vs 원본 재정렬
SQL = {
    "vector": "SELECT id FROM vec_clu ORDER BY emb <-> %(q)s LIMIT 10",
    "halfvec": "SELECT id FROM vec_clu ORDER BY emb::halfvec(128) <-> %(q)s::halfvec(128) LIMIT 10",
    "bit": "SELECT id FROM vec_clu ORDER BY binary_quantize(emb)::bit(128) <~> binary_quantize(%(q)s) LIMIT 10",
}
RERANK = """SELECT id FROM (
              SELECT id, emb FROM vec_clu
              ORDER BY binary_quantize(emb)::bit(128) <~> binary_quantize(%(q)s) LIMIT {r}
            ) c ORDER BY emb <-> %(q)s LIMIT 10"""

rows = []
conn.execute("SET hnsw.ef_search = 100")
for name, sql in SQL.items():
    rows.append({"방식": name, "후보": 10, **pg_bench_sql(conn, sql, "clu")})
for r in [20, 50, 100, 200, 400, 800]:
    conn.execute(f"SET hnsw.ef_search = {max(100, r)}")          # 후보 r 개를 받으려면 ef_search ≥ r (21번 함정)
    rows.append({"방식": "bit + 재정렬", "후보": r, **pg_bench_sql(conn, RERANK.format(r=r), "clu")})
conn.execute("RESET hnsw.ef_search")
res = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1, 1.4]})
base = res[res["방식"] != "bit + 재정렬"]
axes[0].bar(base["방식"], base["recall"], color=colors)
for i, v in enumerate(base["recall"]):
    axes[0].text(i, v, f"{v:.3f}", ha="center", va="bottom")
axes[0].set_ylim(0, 1.08); axes[0].set_ylabel("recall@10"); axes[0].set_title("압축본으로만 검색 (ef_search=100)")
rr = res[res["방식"] == "bit + 재정렬"]
axes[1].plot(rr["후보"], rr["recall"], "-o", color="#FF7043", label="bit 후보 → 원본 재정렬")
axes[1].axhline(base.iloc[0]["recall"], color="#3D484A", ls="--", label="float32 인덱스")
for _, r in rr.iterrows():
    axes[1].annotate(f"{r.recall:.2f}\n{r.ms:.1f}ms", (r["후보"], r.recall), textcoords="offset points", xytext=(0, -30),
                     ha="center", fontsize=8)
axes[1].set_xscale("log"); axes[1].set_xticks(rr["후보"], rr["후보"].astype(str)); axes[1].minorticks_off()
axes[1].set_xlabel("이진 인덱스에서 뽑은 후보 수"); axes[1].set_ylim(0, 1.08)
axes[1].legend(loc="center right"); axes[1].set_title("후보를 넉넉히 뽑고 원본으로 다시 줄 세우기")
plt.tight_layout(); plt.show()
res.round(3)

**관찰:**
- halfvec 은 크기가 절반인데 recall 이 float32 와 거의 같다 — **가장 손쉬운 절감**
- 이진 인덱스만으로는 recall 이 낮다. 1비트로는 가까운 것들끼리 순서를 가리지 못한다
- 후보를 수백 개 뽑아 원본으로 재정렬하면 float32 인덱스 수준까지 회복한다 (13번 PQ + 재정렬과 같은 원리)
- 재정렬 비용은 후보 수에 비례한다. 인덱스는 작게(RAM 절약), 정확도는 원본으로 — 슬라이드 IVF-PQ 의 "원본/고정밀로 재정렬"

### 셀 6 · 뒷정리

In [ ]:
# 셀 6 · 뒷정리
for kind in ["half", "bin"]:
    conn.execute(f"DROP INDEX IF EXISTS vec_clu_{kind}")
print("vec_clu 에는 float32 HNSW 인덱스(vec_clu_hnsw)만 남겼다")